In [ ]:
!pip install groq python-dotenv numpy tqdm datasets math-verify

In [56]:
from groq import Groq
from dotenv import load_dotenv
from datasets import load_dataset, concatenate_datasets

import os
from tqdm import tqdm
import re
import random
import pprint

from typing import List, Dict, Any, Optional

load_dotenv()
random.seed(0)

client = Groq()

MODEL = "llama-3.1-8b-instant"

#### MATH 데이터셋 불러오기

- 평가: `HuggingFaceH4/MATH-500`
- few-shot 예시: `HuggingFaceH4/MATH`의 과목별 train split


In [57]:
# MATH 데이터 난이도 및 과목 필터링
TARGET_LEVELS = [1, 2, 3]

TARGET_SUBJECTS = [
    "Algebra",
    "Intermediate Algebra",
    "Number Theory",
    "Counting & Probability",
]

# 평가용 MATH-500
math_dataset = load_dataset("HuggingFaceH4/MATH-500")
math_test_raw = math_dataset["test"]

# few-shot 예시용 MATH train
TRAIN_CONFIGS = [
    "algebra",
    "intermediate_algebra",
    "number_theory",
    "counting_and_probability",
]

train_parts = []

for config in TRAIN_CONFIGS:
    ds = load_dataset(
        "HuggingFaceH4/MATH",
        config,
        split="train"
    )
    train_parts.append(ds)

math_train_raw = concatenate_datasets(train_parts)

print(sorted(set(math_train_raw["type"])))
print("raw train size:", len(math_train_raw))


['Algebra', 'Counting & Probability', 'Intermediate Algebra', 'Number Theory']
raw train size: 4679


In [58]:
## 데이터셋 전처리
def extract_last_boxed(text: str) -> Optional[str]:
    """문자열에서 마지막 \\boxed{...}의 내용을 추출합니다."""
    if not text:
        return None

    starts = [m.start() for m in re.finditer(r"\\boxed\s*\{", text)]
    if not starts:
        return None

    start = starts[-1]
    open_brace = text.find("{", start)
    depth = 0

    for idx in range(open_brace, len(text)):
        if text[idx] == "{":
            depth += 1
        elif text[idx] == "}":
            depth -= 1
            if depth == 0:
                return text[open_brace + 1:idx].strip()

    return None


def parse_level(level_value) -> Optional[int]:
    match = re.search(r"\d+", str(level_value))
    return int(match.group()) if match else None


def prepare_math_train_row(row):
    return {
        "question": row["problem"],
        "answer": extract_last_boxed(row["solution"]),
        "rationale": row["solution"],
        "subject": row["type"],
        "level_num": parse_level(row["level"]),
    }


math_train = math_train_raw.map(prepare_math_train_row)

math_train = math_train.filter(
    lambda row: (
        row["level_num"] in TARGET_LEVELS
        and row["subject"] in TARGET_SUBJECTS
        and row["answer"] is not None
    )
)

math_test = math_test_raw.filter(
    lambda row: (
        parse_level(row["level"]) in TARGET_LEVELS
        and row["subject"] in TARGET_SUBJECTS
    )
)

print("math_train size:", len(math_train))
print("math_test size:", len(math_test))
print("train levels:", sorted(set(math_train["level_num"])))
print("test levels:", sorted(set(parse_level(x) for x in math_test["level"])))


math_train size: 2132
math_test size: 146
train levels: [1, 2, 3]
test levels: [1, 2, 3]


In [59]:
def generate_response_using_Llama(
        prompt: str,
        model: str = MODEL,
        temperature: float = 0.0
    ):
    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {
                    "role": "system",
                    "content": "You are a helpful assistant that solves math problems."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            model=model,
            temperature=0.0,
            stream=False
        )
        return chat_completion.choices[0].message.content

    except Exception as e:
        print(f"API call error: {str(e)}")
        return None


#### 응답 잘 나오는지 확인하기

In [60]:
response = generate_response_using_Llama(
    prompt="Hello world!",
)
print(response)

Hello world! I'm here to help with any math problems you might have. What's on your mind? Do you have a specific problem you'd like me to solve, or would you like some help with a particular math concept?


#### MATH 데이터셋 확인하기

In [61]:
print("[Question]")
print(math_test[0]["problem"])
print("=" * 100)
print("[Answer]")
print(math_test[0]["answer"])
print("=" * 100)
print("[Solution]")
print(math_test[0]["solution"])


[Question]
If $f(x) = \frac{3x-2}{x-2}$, what is the value of $f(-2) +f(-1)+f(0)$? Express your answer as a common fraction.
[Answer]
\frac{14}{3}
[Solution]
$f(-2)+f(-1)+f(0)=\frac{3(-2)-2}{-2-2}+\frac{3(-1)-2}{-1-2}+\frac{3(0)-2}{0-2}=\frac{-8}{-4}+\frac{-5}{-3}+\frac{-2}{-2}=2+\frac{5}{3}+1=\boxed{\frac{14}{3}}$


#### Utils 함수들
- extract_final_answer: LLM의 응답을 parse하여 최종 결과만 추출 (정답과 비교하기 위해)
- run_benchmark_test: 벤치마크 테스트
- save_final_result: 결과물 제출을 위한 함수

In [62]:
def extract_final_answer(response: str):
    """응답에서 마지막 \\boxed{...} 또는 Answer: 뒤의 답을 추출합니다."""
    if response is None:
        return None

    boxed_answer = extract_last_boxed(response)
    if boxed_answer is not None:
        return boxed_answer

    matches = re.findall(
        r"(?:Final Answer|Answer)\s*:\s*(.+)",
        response,
        re.IGNORECASE
    )
    if matches:
        return matches[-1].strip().strip("$")

    return None


def normalize_math_text(text: Any) -> str:
    text = str(text).strip().strip("$")
    text = text.replace(r"\displaystyle", "")
    text = text.replace(r"\dfrac", r"\frac")
    text = text.replace(r"\tfrac", r"\frac")
    text = text.replace(r"\,", "")
    text = text.replace(" ", "")
    return text.rstrip(".")


try:
    from math_verify import parse, verify
    MATH_VERIFY_AVAILABLE = True
except Exception:
    MATH_VERIFY_AVAILABLE = False


def answers_equivalent(
    correct_answer: str,
    predicted_answer: Optional[str]
) -> bool:
    if predicted_answer is None:
        return False

    if MATH_VERIFY_AVAILABLE:
        try:
            correct_parsed = parse(f"${correct_answer}$")
            predicted_parsed = parse(f"${predicted_answer}$")

            if verify(correct_parsed, predicted_parsed):
                return True
        except Exception:
            pass

    return (
        normalize_math_text(correct_answer)
        == normalize_math_text(predicted_answer)
    )


print("math-verify available:", MATH_VERIFY_AVAILABLE)


math-verify available: True


In [63]:
### 수정해도 됩니다!
import time


def generate_with_retry(prompt, model=MODEL, temperature=0.0,
                        max_retries=5, base_delay=8.0):
    for attempt in range(max_retries):
        r = generate_response_using_Llama(prompt, model=model,
                                          temperature=temperature)
        if r is not None:
            return r
        delay = base_delay * (2 ** attempt)
        print(f"  [retry {attempt+1}/{max_retries}] waiting {delay:.0f}s...")
        time.sleep(delay)
    return None


def majority_vote(answers):
    """math_verify 동치판정으로 묶어서 다수결. 동률이면 먼저 나온 쪽."""
    groups = []  # [대표답, 표수, 등장순서]
    for a in answers:
        if a is None:
            continue
        for g in groups:
            if answers_equivalent(g[0], a):
                g[1] += 1
                break
        else:
            groups.append([a, 1, len(groups)])
    if not groups:
        return None, []
    groups.sort(key=lambda g: (-g[1], g[2]))
    return groups[0][0], [(g[0], g[1]) for g in groups]


def run_benchmark_test(dataset, prompt, model=MODEL, num_samples=50,
                       VERBOSE=False, k=3, temperature=0.7):
    correct = total = api_errors = saved_calls = 0
    results = []

    for i in tqdm(range(min(num_samples, len(dataset)))):
        question = dataset[i]["problem"]
        correct_answer = str(dataset[i]["answer"]).strip()
        final_prompt = prompt.replace("{question}", question)

        preds, responses = [], []
        for j in range(k):
            # 조기 종료: 앞 두 샘플이 일치하면 나머지 생략
            if j >= 2 and len(preds) >= 2 and preds[0] is not None \
               and answers_equivalent(preds[0], preds[1]):
                saved_calls += k - j
                break
            temp = 0.0 if (k == 1) else temperature
            r = generate_with_retry(final_prompt, model=model, temperature=temp)
            if r is None:
                api_errors += 1
            responses.append(r)
            preds.append(extract_final_answer(r) if r else None)

        predicted_answer, vote_detail = majority_vote(preds)
        is_correct = answers_equivalent(correct_answer, predicted_answer)

        if VERBOSE:
            print("=" * 50)
            print(f"Votes: {vote_detail}")
            print(f"Correct Answer: {correct_answer}")
            print(f"Predicted Answer: {predicted_answer} | {is_correct}")

        correct += int(is_correct)
        total += 1

        results.append({
            "question": question,
            "correct_answer": correct_answer,
            "predicted_answer": predicted_answer,
            "correct": is_correct,
            "subject": dataset[i]["subject"],
            "level": parse_level(dataset[i]["level"]),
            "response": responses[-1] if responses else None,
            "all_responses": responses,
            "all_predictions": preds,
            "vote_detail": vote_detail,
            "api_error": all(r is None for r in responses),
        })

        if total % 5 == 0:
            print(f"Progress: [{total}/{min(num_samples, len(dataset))}]")
            print(f"Current Acc.: [{correct/total:.2%}]")

    accuracy = correct / total if total else 0.0
    parse_fails = sum(1 for r in results
                      if not r["api_error"] and r["predicted_answer"] is None)
    print(f"\n[summary] acc={accuracy:.2%} | k={k} | api_error={api_errors} "
          f"| parse_fail={parse_fails} | saved_calls={saved_calls}")
    return results, accuracy

In [64]:
def save_final_result(
    results: List[Dict[str, Any]],
    accuracy: float,
    filename: str
) -> None:
    result_str = f"====== ACCURACY: {accuracy} ======\n\n"
    result_str += "[Details]\n"

    for idx, result in enumerate(results):
        result_str += f"Question {idx + 1}: {result['question']}\n"
        result_str += f"Subject: {result['subject']}\n"
        result_str += f"Level: {result['level']}\n"
        result_str += f"Correct Answer: {result['correct_answer']}\n"
        result_str += f"Predicted Answer: {result['predicted_answer']}\n"
        result_str += f"Correct: {result['correct']}\n\n"

    with open(filename, "w", encoding="utf-8") as f:
        f.write(result_str)


#### 1. Direct Prompting with few-shot examples

In [65]:
def select_few_shot_indices(num_examples: int) -> List[int]:
    if not 0 <= num_examples <= len(math_train):
        raise ValueError("num_examples must be between 0 and len(math_train).")

    # A dedicated seed makes Direct and CoT use identical examples per shot.
    return random.Random(0).sample(range(len(math_train)), num_examples)


def construct_direct_prompt(num_examples: int = 3) -> str:
    train_dataset = math_train
    sampled_indices = select_few_shot_indices(num_examples)

    prompt = (
        "Instruction:\n"
        "Solve the following mathematical question and generate ONLY the final answer "
        "after the tag 'Answer:' without any rationale. "
        "Use valid mathematical notation.\n"
    )

    for idx, i in enumerate(sampled_indices):
        cur_question = train_dataset[i]["question"]
        cur_answer = train_dataset[i]["answer"]

        prompt += f"\n[Example {idx + 1}]\n"
        prompt += f"Question:\n{cur_question}\n"
        prompt += f"Answer: {cur_answer}\n"

    prompt += "\nQuestion:\n{question}\nAnswer:"

    return prompt


In [11]:
### 어떤 방식으로 저장되는지 확인해보세요!

PROMPT = construct_direct_prompt(3)
VERBOSE = False

results, accuracy = run_benchmark_test(
    dataset=math_test,
    prompt=PROMPT,
    VERBOSE=VERBOSE,
    num_samples=10
)
save_final_result(results, accuracy, "example.txt")
print(f"Direct 3-shot demo accuracy: {accuracy:.2%}")


 50%|█████     | 5/10 [00:03<00:02,  1.73it/s]

Progress: [5/10]
Current Acc.: [80.00%]


100%|██████████| 10/10 [00:05<00:00,  1.67it/s]

Progress: [10/10]
Current Acc.: [70.00%]
Direct 3-shot demo accuracy: 70.00%


In [12]:
DIRECT_RESULTS = {}
DIRECT_ACCURACIES = {}

for shot in (0, 3, 5):
    direct_prompt = construct_direct_prompt(num_examples=shot)
    results, accuracy = run_benchmark_test(
        dataset=math_test,
        prompt=direct_prompt,
        num_samples=50,
        VERBOSE=False
    )

    filename = f"direct_prompting_{shot}.txt"
    save_final_result(results, accuracy, filename)

    DIRECT_RESULTS[shot] = results
    DIRECT_ACCURACIES[shot] = accuracy
    print(f"Direct {shot}-shot accuracy: {accuracy:.2%} | saved: {filename}")

 10%|█         | 5/50 [00:01<00:12,  3.75it/s]

Progress: [5/50]
Current Acc.: [20.00%]


 20%|██        | 10/50 [00:03<00:13,  2.96it/s]

Progress: [10/50]
Current Acc.: [20.00%]


 30%|███       | 15/50 [00:08<00:49,  1.40s/it]

...

Progress: [40/50]
Current Acc.: [42.50%]


 90%|█████████ | 45/50 [04:04<00:26,  5.27s/it]

Progress: [45/50]
Current Acc.: [40.00%]


100%|██████████| 50/50 [04:54<00:00,  5.90s/it]

Progress: [50/50]
Current Acc.: [40.00%]
Direct 5-shot accuracy: 40.00% | saved: direct_prompting_5.txt


#### 2. Chain-of-Thought Prompting with few-shot examples

```text
[Question]
Janet’s ducks lay 16 eggs per day
 She eats three for breakfast every morning and bakes muffins for her friends every day with four
 She sells the remainder at the farmers' market daily for $2 per fresh duck egg
 How much in dollars does she make every day at the farmers' market?
====================================================================================================
[Answer]
Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18
```

[Answer] 아래의 정답을 도출하는 과정을 예시로 달아주면 CoT의 few shot이 됩니다.

In [9]:
def construct_CoT_prompt(num_examples: int = 3) -> str:
    train_dataset = math_train
    sampled_indices = select_few_shot_indices(num_examples)

    prompt = (
        "Instruction:\n"
        "Solve the mathematical question step by step.\n"
        "Show the reasoning needed to reach the answer, then end with exactly one line "
        "in the format Answer: <final answer>.\n"
    )

    for idx, i in enumerate(sampled_indices):
        cur_question = train_dataset[i]["question"]
        cur_rationale = train_dataset[i]["rationale"]
        cur_answer = train_dataset[i]["answer"]

        prompt += f"\n[Example {idx + 1}]\n"
        prompt += f"Question:\n{cur_question}\n"
        prompt += f"Solution:\n{cur_rationale}\n"
        prompt += f"Answer: {cur_answer}\n"

    prompt += "\nQuestion:\n{question}\nSolution:\n"

    return prompt


In [10]:
COT_RESULTS = {}
COT_ACCURACIES = {}

for shot in (0, 3, 5):
    cot_prompt = construct_CoT_prompt(num_examples=shot)
    results, accuracy = run_benchmark_test(
        dataset=math_test,
        prompt=cot_prompt,
        num_samples=50,
        VERBOSE=False
    )

    filename = f"CoT_prompting_{shot}.txt"
    save_final_result(results, accuracy, filename)

    COT_RESULTS[shot] = results
    COT_ACCURACIES[shot] = accuracy
    print(f"CoT {shot}-shot accuracy: {accuracy:.2%} | saved: {filename}")

  0%|          | 0/50 [00:00<?, ?it/s]

Progress: [5/50]
Current Acc.: [60.00%]


 16%|█▌        | 8/50 [00:05<00:26,  1.58it/s]

Progress: [10/50]
Current Acc.: [70.00%]


 30%|███       | 15/50 [00:55<03:07,  5.36s/it]

...

Progress: [40/50]
Current Acc.: [75.00%]


 90%|█████████ | 45/50 [09:38<00:57, 11.56s/it]

Progress: [45/50]
Current Acc.: [73.33%]


100%|██████████| 50/50 [10:28<00:00, 12.57s/it]

Progress: [50/50]
Current Acc.: [74.00%]

[summary] acc=74.00% | api_error=0 | parse_fail=2
CoT 5-shot accuracy: 74.00% | saved: CoT_prompting_5.txt


#### 3. Construct your prompt + few shot examples
목표: 본인만의 프롬프트를 통해 정답률을 더 끌어올리기!
- 세션때 배운 내용을 활용하거나 본인만의 풀이 과정을 만드는 등 자유롭게 진행해주시면 됩니다.
- 정답률은 Direct Prompting, CoT Prompting을 한 결과보다 높으면 됩니다. (0-shot, 3-shot, 5-shot 각각에서 모두 Direct Prompting과 CoT Prompting보다 높은 정답률을 달성하지 않더라도 감안하여 채점하겠습니다. 종합적으로 비교했을 때 본인이 설계한 프롬프트가 전반적으로 더 높은 성능을 보이는지를 기준으로 보겠습니다.)

In [66]:
def construct_my_prompt(num_examples: int = 3) -> str:
    train_dataset = math_train
    sampled_indices = select_few_shot_indices(num_examples)
    few_shot_text = ""

    for idx, i in enumerate(sampled_indices):
        cur_question = train_dataset[i]["question"]
        cur_rationale = train_dataset[i]["rationale"]
        cur_answer = train_dataset[i]["answer"]

        few_shot_text += rf"""
[Example {idx + 1}]
Question:
{cur_question}

Solution:
{cur_rationale}

Answer: \boxed{{{cur_answer}}}
"""

    rules = r"""Rules:
- Begin with a brief plan, then solve using the shortest valid method.
  Show the essential intermediate steps.
- Once you obtain a result, stop. Do not look for an alternative form of the
  same value, do not present a second method, and do not re-derive a quantity
  you already computed. Never start a sentence with "However" after reaching
  a result.
- If the question asks for the smallest, least, greatest, or largest value,
  derive the constraints first, then test candidates in order and take the
  first one satisfying all of them.
- Track the exact quantity requested (the quotient, not the remainder;
  the count, not the value).
- Use exact values in LaTeX. Simplify the final answer.
- End with exactly one line, with nothing after it:
Answer: \boxed{<final answer>}"""

    reminder = r"""Reminder: stop as soon as you have a result; no second method,
no alternative forms. End with exactly one line:
Answer: \boxed{<final answer>}"""

    prompt = rf"""Instruction:
Solve the mathematical question accurately.

{rules}

{few_shot_text}

{reminder}

Question:
{{question}}
Solution:
"""

    return prompt

In [ ]:
MY_RESULTS, MY_ACCURACIES = {}, {}

for shot in (0, 3):
    my_prompt = construct_my_prompt(num_examples=shot)
    results, accuracy = run_benchmark_test(
        dataset=math_test, prompt=my_prompt,
        num_samples=50, k=3, temperature=0.7
    )
    MY_RESULTS[shot] = results
    MY_ACCURACIES[shot] = accuracy
    save_final_result(results, accuracy, f"My_prompting_{shot}.txt")
    print(f"My {shot}-shot: {accuracy:.2%}")

  0%|          | 0/50 [00:00<?, ?it/s]

API call error: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kz0s6r92fshvb9x2yw89pwys` service tier `on_demand` on tokens per day (TPD): Limit 500000, Used 499483, Requested 763. Please try again in 42.5088s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
  [retry 1/5] waiting 8s...


  2%|▏         | 1/50 [02:17<1:52:36, 137.88s/it]

API call error: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01kz0s6r92fshvb9x2yw89pwys` service tier `on_demand` on tokens per day (TPD): Limit 500000, Used 500000, Requested 498. Please try again in 1m26.054399999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
  [retry 1/5] waiting 8s...


### 여기서부터 다시

In [ ]:
## 

### 보고서 작성하기
#### 아래의 내용이 포함되면 됩니다!

1. Direct Prompting, CoT Prompting, My Prompting을 0 shot, 3 shot 정답률을 표로 보여주세요.
2. CoT Prompting이 Direct Prompting에 비해 왜 좋을 수 있는지에 대해서 서술해주세요.
3. 본인이 작성한 프롬프트 기법에 대해서 설명하고 CoT에 비해서 왜 더 좋을 수 있는지에 대해서 설명해주세요.
4. 위 내용들을 `PROMPTING.md`에 보고서로 작성해주세요.
